In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


25/12/07 21:33:47 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/07 21:33:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-eddfdd3f-c1bf-4fda-9214-1a17fbb57d78;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


Cell 1 – Intro
# %% [markdown]
# # ML – Claims High-Cost Risk Model
#
# **Goal:** Train and evaluate Spark ML models to predict *high-cost claims* using
# the curated Gold feature table `ft_claims_risk_split`.
#
# Label: `Is_High_Cost` (0 = normal, 1 = high cost)
#
# Steps:
# 1. Load Gold feature table
# 2. Clean + train/test split
# 3. Build Spark ML pipeline (index + encode + assemble)
# 4. Train several models (LR, RF, GBT)
# 5. Persist the best model
# 6. Score sample claims with high-cost probability


# Cell 2 – Imports & config

In [2]:
# Cell 2 — Imports & config

from pyspark.sql import functions as F
from pyspark.sql import DataFrame

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import (
    RandomForestClassifier,
    LogisticRegression,
    GBTClassifier,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

ACCOUNT        = "clientdatastorage"
GOLD_CONTAINER = "golddata"

FT_CLAIMS_RISK_SPLIT_PATH = (
    f"abfss://{GOLD_CONTAINER}@{ACCOUNT}.dfs.core.windows.net/ft_claims_risk_split"
)

MODEL_BASE_PATH = (
    f"abfss://{GOLD_CONTAINER}@{ACCOUNT}.dfs.core.windows.net/models"
)
MODEL_NAME = "claims_high_cost_best"
MODEL_PATH = f"{MODEL_BASE_PATH}/{MODEL_NAME}"

print("Feature table path:", FT_CLAIMS_RISK_SPLIT_PATH)
print("Model path        :", MODEL_PATH)


Feature table path: abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split
Model path        : abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_high_cost_best


# Cell 3 – Load feature table

In [3]:
# Cell 3 — Load ft_claims_risk_split

claims_all = (
    spark.read
         .format("delta")
         .load(FT_CLAIMS_RISK_SPLIT_PATH)
)

print("Total rows in ft_claims_risk_split:", claims_all.count())
claims_all.printSchema()
claims_all.show(5, truncate=False)


25/12/07 21:36:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total rows in ft_claims_risk_split: 558211
root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Type_Code: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Days_To_Settle: integer (nullable = true)
 |-- High_Cost_Claim_Flag: integer (nullable = true)
 |-- Claim_Fraud_Label: integer (nullable = true)
 |-- Provider_Fraud_Label: integer (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_date_valid: integer (nullable = true)
 |-- Is_Fraudulent_Claim: integer (nullable = true)
 |-- Is_High_Cost: integer (nullable = true)
 |-- dataset_split: string (nullable = true)



+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Type_Code|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Days_To_Settle|High_Cost_Claim_Flag|Claim_Fraud_Label|Provider_Fraud_Label|Provider_Risk_Tier|dq_money_valid|dq_date_valid|Is_Fraudulent_Claim|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |TRAVEL         |Settled     |244.07511199

# Cell 4 – Filter label + train/test

In [4]:
# Cell 4 — Use Is_High_Cost as label

LABEL_COL = "Is_High_Cost"

if LABEL_COL not in claims_all.columns:
    raise ValueError(f"Expected label column '{LABEL_COL}' not found in ft_claims_risk_split")

claims_labeled = (
    claims_all
      .filter(F.col(LABEL_COL).isNotNull())
      .withColumn(LABEL_COL, F.col(LABEL_COL).cast("double"))
)

# Optional: keep only dq-valid rows
claims_labeled = claims_labeled.filter(
    (F.col("dq_money_valid") == 1) & (F.col("dq_date_valid") == 1)
)

print("Rows with non-null label & dq-valid:", claims_labeled.count())

train_df = claims_labeled.filter(F.col("dataset_split") == "train")
test_df  = claims_labeled.filter(F.col("dataset_split") == "test")

print("Train rows:", train_df.count())
print("Test rows :", test_df.count())


Rows with non-null label & dq-valid: 558211


Train rows: 446102
Test rows : 112109


# Cell 5 – Feature columns & null handling

In [5]:
# Cell 5 — Feature selection & null handling

numeric_candidates = [
    "Claim_Amount_GBP",
    "Payout_GBP",
    "Payout_to_Amount_Ratio",
    "Days_To_Settle",
]

categorical_candidates = [
    "Claim_Type_Name",
    "Claim_Status",
    "Claim_Type_Code",
    "Provider_Risk_Tier",
    "Provider_ID",
]

cols = claims_labeled.columns

numeric_cols = [c for c in numeric_candidates if c in cols]
cat_cols     = [c for c in categorical_candidates if c in cols]

print("Numeric features   :", numeric_cols)
print("Categorical features:", cat_cols)

id_cols = [c for c in ["Claim_ID", "Member_Key", "Provider_ID"] if c in cols]

def prep_nulls(df: DataFrame) -> DataFrame:
    d = df
    for c in numeric_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit(0.0)).otherwise(F.col(c)))
    for c in cat_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit("Unknown")).otherwise(F.col(c)))
    return d


Numeric features   : ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle']
Categorical features: ['Claim_Type_Name', 'Claim_Status', 'Claim_Type_Code', 'Provider_Risk_Tier', 'Provider_ID']


# Cell 6 – Build pipeline skeleton

In [6]:
# Cell 6 — Indexers, encoders, assembler

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep",
    )
    for c in cat_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_oh",
        handleInvalid="keep",
    )
    for c in cat_cols
]

feature_cols = numeric_cols + [f"{c}_oh" for c in cat_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep",
)

print("Feature columns in assembler:", feature_cols)


Feature columns in assembler: ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle', 'Claim_Type_Name_oh', 'Claim_Status_oh', 'Claim_Type_Code_oh', 'Provider_Risk_Tier_oh', 'Provider_ID_oh']


# Cell 7 – Train & eval helper

In [7]:
# Cell 7 — Train & evaluate helper

def train_and_eval(pipeline: Pipeline, train_df: DataFrame, test_df: DataFrame, model_name: str):
    print(f"\n===== Training {model_name} =====")
    model = pipeline.fit(train_df)

    print(f"Scoring test set with {model_name} ...")
    pred = model.transform(test_df)

    evaluator_roc = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )
    evaluator_pr = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR",
    )

    roc_auc = evaluator_roc.evaluate(pred)
    pr_auc  = evaluator_pr.evaluate(pred)

    preds = pred.select(
        LABEL_COL,
        F.col("prediction").cast("double").alias("prediction")
    )

    tp = preds.filter((F.col(LABEL_COL) == 1) & (F.col("prediction") == 1)).count()
    tn = preds.filter((F.col(LABEL_COL) == 0) & (F.col("prediction") == 0)).count()
    fp = preds.filter((F.col(LABEL_COL) == 0) & (F.col("prediction") == 1)).count()
    fn = preds.filter((F.col(LABEL_COL) == 1) & (F.col("prediction") == 0)).count()

    total = tp + tn + fp + fn
    accuracy  = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    metrics = {
        "model": model_name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

    print(f"ROC AUC: {roc_auc:.4f} | PR AUC: {pr_auc:.4f} | "
          f"Acc: {accuracy:.4f} | Prec: {precision:.4f} | Rec: {recall:.4f} | F1: {f1:.4f}")

    return model, metrics, pred


# Cell 8 – Define models & train

In [8]:
# Cell 8 — Train LR, RF, GBT for high-cost label

train_pre = prep_nulls(train_df)
test_pre  = prep_nulls(test_df)

rf = RandomForestClassifier(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=8,
    seed=42,
)

lr = LogisticRegression(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0,
)

"""

gbt = GBTClassifier(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    maxIter=80,
    maxDepth=6,
    stepSize=0.1,
    seed=42,
)

"""

pipelines = {
    "RandomForest": Pipeline(stages=indexers + encoders + [assembler, rf]),
    "LogisticRegression": Pipeline(stages=indexers + encoders + [assembler, lr]),
    #"GBTClassifier": Pipeline(stages=indexers + encoders + [assembler, gbt]),
}

results = []
trained_models = {}

for name, pipe in pipelines.items():
    model, metrics, _ = train_and_eval(pipe, train_pre, test_pre, name)
    results.append(metrics)
    trained_models[name] = model

print("\n===== Summary of models =====")
for m in results:
    print(m)



===== Training RandomForest =====


25/12/07 21:40:34 WARN MemoryStore: Not enough space to cache rdd_163_2 in memory! (computed 321.9 MiB so far)
25/12/07 21:40:34 WARN BlockManager: Persisting block rdd_163_2 to disk instead.
25/12/07 21:41:04 WARN MemoryStore: Not enough space to cache rdd_163_2 in memory! (computed 321.9 MiB so far)
25/12/07 21:41:24 WARN MemoryStore: Not enough space to cache rdd_163_2 in memory! (computed 321.9 MiB so far)
25/12/07 21:41:46 WARN MemoryStore: Not enough space to cache rdd_163_2 in memory! (computed 321.9 MiB so far)
25/12/07 21:42:10 WARN DAGScheduler: Broadcasting large task binary with size 1026.5 KiB
25/12/07 21:42:10 WARN MemoryStore: Not enough space to cache rdd_163_2 in memory! (computed 321.9 MiB so far)
25/12/07 21:42:36 WARN DAGScheduler: Broadcasting large task binary with size 1066.4 KiB
25/12/07 21:42:37 WARN MemoryStore: Not enough space to cache rdd_163_2 in memory! (computed 321.9 MiB so far)
25/12/07 21:43:05 WARN DAGScheduler: Broadcasting large task binary with si

Scoring test set with RandomForest ...


25/12/07 21:44:38 WARN DAGScheduler: Broadcasting large task binary with size 1059.9 KiB
25/12/07 21:44:44 WARN DAGScheduler: Broadcasting large task binary with size 1059.9 KiB


ROC AUC: 0.9915 | PR AUC: 0.9814 | Acc: 0.8972 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000

===== Training LogisticRegression =====


25/12/07 21:45:12 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/12/07 21:45:12 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


Scoring test set with LogisticRegression ...


ROC AUC: 0.9988 | PR AUC: 0.9911 | Acc: 0.9640 | Prec: 0.9985 | Rec: 0.6506 | F1: 0.7878

===== Summary of models =====
{'model': 'RandomForest', 'roc_auc': 0.9915419694992553, 'pr_auc': 0.9813793342024076, 'accuracy': 0.8972071822958014, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'tp': 0, 'tn': 100585, 'fp': 0, 'fn': 11524}
{'model': 'LogisticRegression', 'roc_auc': 0.9988387591561941, 'pr_auc': 0.9910518216699881, 'accuracy': 0.9639814823073972, 'precision': 0.9985348961108151, 'recall': 0.6505553627212773, 'f1': 0.7878310214375789, 'tp': 7497, 'tn': 100574, 'fp': 11, 'fn': 4027}


# Cell 9 – Choose best model & save

In [9]:
# Cell 9 — Select best model (ROC AUC) and save

best = sorted(results, key=lambda r: r["roc_auc"], reverse=True)[0]
best_name = best["model"]
best_model = trained_models[best_name]

print(f"\nBest model by ROC AUC: {best_name} (ROC AUC={best['roc_auc']:.4f})")

best_model.write().overwrite().save(MODEL_PATH)
print(f"✅ Saved best high-cost model to: {MODEL_PATH}")



Best model by ROC AUC: LogisticRegression (ROC AUC=0.9988)


✅ Saved best high-cost model to: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_high_cost_best


# Cell 10 – Load & score sample claims

In [10]:
# Cell 10 — Load saved model and score some claims

loaded_model = PipelineModel.load(MODEL_PATH)

sample_to_score = (
    test_df
        .orderBy(F.rand())
        .limit(20)
)

scored_raw = loaded_model.transform(prep_nulls(sample_to_score))

scored = (
    scored_raw
        .withColumn("prob_arr", vector_to_array("probability"))
        .withColumn("high_cost_prob", F.col("prob_arr")[1])
        .select(
            *[c for c in id_cols if c in scored_raw.columns],
            LABEL_COL,
            "high_cost_prob",
            "prediction",
            "Claim_Type_Name",
            "Claim_Status",
            "Claim_Amount_GBP",
            "Payout_GBP",
            "Payout_to_Amount_Ratio",
            "Provider_Risk_Tier",
        )
)

scored.printSchema()
scored.show(truncate=False)


root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Is_High_Cost: double (nullable = true)
 |-- high_cost_prob: double (nullable = true)
 |-- prediction: double (nullable = false)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)



+---------+----------+-----------+------------+--------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+
|Claim_ID |Member_Key|Provider_ID|Is_High_Cost|high_cost_prob      |prediction|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|
+---------+----------+-----------+------------+--------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+
|CLM192380|BENE107812|PRV52822   |0.0         |0.019872146658165613|0.0       |Hospital       |Settled     |104.05727114988879|80.0      |0.7688073991942801    |Low risk          |
|CLM294683|BENE82244 |PRV55489   |0.0         |0.02564906397022726 |0.0       |Outpatient     |Settled     |50.54527268006713 |40.0      |0.79136975386769      |Low risk          |
|CLM687161|BENE51870 |PRV53294   |0.0         |0.030245295424701668|0.0       |Outpatient     |